# E-Dada - Modern AI - Final Project - Reinforcement Learning

# Task3_Atari_RL_Breakout Notebook

In [ ]:
# Task 3 – Install Atari Breakout dependencies
!pip install -q \
  "gymnasium[atari,accept-rom-license]" \
  "ale-py" \
  "shimmy[atari]" \
  "tensorboard"

In [ ]:
# Install necessary dependencies
!pip install "gymnasium[atari]"
!pip install stable_baselines3

In [ ]:
import gymnasium as gym
import stable_baselines3
print("Gym:", gym.__version__)
print("SB3:", stable_baselines3.__version__)

In [ ]:
# Imports and log / video folders
# Atari Breakout with DQN

import os
import glob
import numpy as np
import gymnasium as gym

import ale_py

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder

# Register ALE environments so "ALE/Breakout-v5" exists
gym.register_envs(ale_py)

# Folders for TensorBoard and videos
LOG_DIR = "./breakout_tensorboard/"
VIDEO_DIR = "./videos"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

print("Folders ready:", LOG_DIR, VIDEO_DIR)

In [ ]:
# Create Breakout training environment
# Create vectorized Breakout training env
ENV_ID = "ALE/Breakout-v5"   # Gymnasium Atari ID

env = make_atari_env(ENV_ID, n_envs=4, seed=0)
env = VecFrameStack(env, n_stack=4)

print("Training env:", ENV_ID)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

In [ ]:
# Train DQN on ALE/Breakout-v5
# Train DQN on Breakout
from stable_baselines3 import DQN

TRAIN_STEPS = 1_000_000   # you can increase later if you want

model = DQN(
    "CnnPolicy",
    env,                         # the vectorized ALE/Breakout-v5 env you just created
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=50_000,
    batch_size=32,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    target_update_interval=10_000,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    verbose=1,
    tensorboard_log=LOG_DIR,     # "./breakout_tensorboard/"
)

model.learn(
    total_timesteps=TRAIN_STEPS,
    tb_log_name="DQN_Breakout_1",

)


model.save("model_Breakout_DQN")
env.close()
print("Saved model as model_Breakout_DQN.zip")

In [ ]:
# View Breakout DQN training in TensorBoard
%load_ext tensorboard
%tensorboard --logdir ./breakout_tensorboard/

In [ ]:
# Zip the Breakout TensorBoard folder
!zip -r breakout_tensorboard.zip breakout_tensorboard

In [ ]:
# Create the Breakout video
from stable_baselines3 import DQN
import gymnasium as gym
import ale_py
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from gymnasium.wrappers import RecordVideo
import numpy as np
import os

gym.register_envs(ale_py)

ENV_ID = "ALE/Breakout-v5"

eval_env = make_atari_env(ENV_ID, n_envs=1, seed=0)
eval_env = VecFrameStack(eval_env, n_stack=4)

model = DQN.load("model_Breakout_DQN.zip")
print("Loaded Breakout model.")

In [ ]:
# Record multiple eval episodes and keep ONLY the best one
from stable_baselines3.common.vec_env import VecVideoRecorder
import glob, os

num_eval_episodes = 20       # or 30, 50, etc
max_ep_timesteps   = 1000    # same as your training/eval setting

best_reward = -float("inf")
best_video_path = None

for ep in range(num_eval_episodes):
    print(f"\n=== Evaluation episode {ep+1}/{num_eval_episodes} ===")

    # Re-create eval env (same wrappers as in training)
    eval_env = make_atari_env(ENV_ID, n_envs=1, seed=ep)
    eval_env = VecFrameStack(eval_env, n_stack=4)

    # Wrap with VecVideoRecorder – each episode gets its own prefix
    video_prefix = f"Task3_Breakout_DQN_ep{ep}"
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder="./videos",
        name_prefix=video_prefix,
        record_video_trigger=lambda step: step == 0,  # record from first step
        video_length=max_ep_timesteps,
    )

    # Run one evaluation episode
    obs = eval_env.reset()
    episode_reward = 0.0

    for t in range(max_ep_timesteps):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)

        episode_reward += float(rewards[0])

        if dones[0]:
            print(f"Episode finished at step {t} with reward {episode_reward:.2f}")
            break

    eval_env.close()  # flush & finish writing the mp4

    # Locate the just-created video file for this episode
    mp4_files = glob.glob(f"./videos/{video_prefix}*.mp4")
    if not mp4_files:
        print("⚠️ No video file found for this episode.")
        continue

    video_path = mp4_files[0]

    # Update best if this episode is better
    if episode_reward > best_reward:
        best_reward = episode_reward
        best_video_path = video_path

    print(f"Episode {ep} reward = {episode_reward:.2f}, file = {video_path}")

# After all episodes, rename the best one to a nice final name
if best_video_path is not None:
    final_name = "./videos/Task3_Breakout_DQN_BestEpisode.mp4"
    os.replace(best_video_path, final_name)
    print(f"\n✅ Best eval reward = {best_reward:.2f}")
    print(f"Best episode video saved as: {final_name}")
else:
    print("❌ No best video found (something went wrong).")

In [ ]:
# Record the Breakout evaluation video
from stable_baselines3.common.vec_env import VecVideoRecorder

max_ep_timesteps = 1000  # same as you used before

# Re-create eval env (same wrappers as in training)
eval_env = make_atari_env(ENV_ID, n_envs=1, seed=0)
eval_env = VecFrameStack(eval_env, n_stack=4)

# Wrap with VecVideoRecorder (this will save an .mp4 in ./videos)
eval_env = VecVideoRecorder(
    eval_env,
    video_folder="./videos",
    name_prefix="Task3_Breakout_DQN_LastEpisode",
    record_video_trigger=lambda step: step == 0,  # record from the first episode
    video_length=max_ep_timesteps,
)

# Run one evaluation episode
obs = eval_env.reset()
episode_reward = 0.0

for t in range(max_ep_timesteps):
    action, _ = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = eval_env.step(action)

    episode_reward += float(rewards[0])

    if dones[0]:
        print(f"Episode finished at step {t}")
        break

print(f"Final episode reward: {episode_reward:.2f}")
eval_env.close()

In [ ]:
# Clean up the video filename
import os, glob

mp4_files = glob.glob("videos/Task3_Breakout_DQN_LastEpisode*.mp4")
print("MP4 files:", mp4_files)

if mp4_files:
    old_name = mp4_files[0]
    new_name = "videos/Task3_Breakout_DQN_LastEpisode.mp4"
    os.rename(old_name, new_name)
    print("Renamed to:", new_name)
else:
    print("No Task3 video found.")

In [ ]:
# Zip the Breakout TensorBoard folder
!zip -r breakout_tensorboard.zip breakout_tensorboard

In [ ]:
!zip -r videos_task3_breakout.zip videos